# Dimensionality Reduction & Feature Selection

## Feature Selection: Filter Methods

Filter methods rank features by statistical tests independent of the model. Fast but ignore feature interactions.

```python title="example1.py"
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.datasets import load_iris
import pandas as pd

# Load data
X, y = load_iris(return_X_y=True)
feature_names = load_iris().feature_names

# Univariate statistical test (ANOVA F-test)
selector = SelectKBest(score_func=f_classif, k=2)
X_selected = selector.fit_transform(X, y)

# Get selected feature names
selected_features = [feature_names[i] for i in selector.get_support(indices=True)]
print(f"Selected features: {selected_features}")

# Feature scores
scores = selector.scores_
feature_scores = pd.DataFrame({
    'feature': feature_names,
    'score': scores
}).sort_values('score', ascending=False)
print(feature_scores)

# Mutual information (captures non-linear relationships)
selector_mi = SelectKBest(score_func=mutual_info_classif, k=2)
X_selected_mi = selector_mi.fit_transform(X, y)
print(f"MI-selected features: {[feature_names[i] for i in selector_mi.get_support(indices=True)]}")
```

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/scikit-learn-machine-learning/mod-28.ipynb)

```
Selected features: ['petal length (cm)', 'petal width (cm)']
feature      score
petal width   1711.0
petal length  1530.0
```

## Feature Selection: Wrapper Methods

Wrapper methods use model performance to select features. More expensive but capture feature interactions.

```python title="example2.py"
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# Recursive Feature Elimination (RFE)
model = RandomForestClassifier(n_estimators=100, random_state=42)
rfe = RFE(estimator=model, n_features_to_select=2, step=1)
X_rfe = rfe.fit_transform(X, y)

# Selected features
selected_features_rfe = [feature_names[i] for i in rfe.get_support(indices=True)]
print(f"RFE selected features: {selected_features_rfe}")

# Feature importance from Random Forest
model.fit(X, y)
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)
print(feature_importance_df)
```

## Dimensionality Reduction: PCA

PCA (Principal Component Analysis) creates new uncorrelated features (principal components) that capture maximum variance. Useful for visualization and noise reduction.

```python title="example3.py"
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# PCA: reduce to 2 components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Explained variance
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Cumulative variance: {pca.explained_variance_ratio_.cumsum()}")

# Visualize
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', alpha=0.6)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('PCA Projection')
plt.colorbar(scatter)
plt.show()

# Determine optimal components
pca_full = PCA()
pca_full.fit(X)
cumsum = pca_full.explained_variance_ratio_.cumsum()
n_components = (cumsum >= 0.95).argmax() + 1  # 95% variance
print(f"Components for 95% variance: {n_components}")
```

> **💡 Tip:** Use PCA when you have many correlated features. Use feature selection when interpretability matters (original features are meaningful).

## t-SNE for Visualization

t-SNE (t-Distributed Stochastic Neighbor Embedding) is excellent for visualizing high-dimensional data in 2D/3D, but not suitable for feature extraction (non-deterministic, computationally expensive).

```python title="example4.py"
from sklearn.manifold import TSNE

# t-SNE: reduce to 2D for visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='viridis', alpha=0.6)
plt.title('t-SNE Visualization')
plt.colorbar(scatter)
plt.show()
```

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the main advantage of wrapper methods over filter methods?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387800001" value="0">
      <span>They are faster</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387800001" value="1">
      <span>They capture feature interactions and model-specific performance</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387800001" value="2">
      <span>They work with any data type</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387800001" value="3">
      <span>They reduce overfitting</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ When should you use t-SNE?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387800002" value="0">
      <span>For feature extraction in production models</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387800002" value="1">
      <span>For reducing computational cost</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387800002" value="2">
      <span>For visualizing high-dimensional data in 2D/3D</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387800002" value="3">
      <span>For handling missing values</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>